1) Import libraries

In [1]:
import pandas as pd
import requests
import duckdb
import io

2) Set Script Parameters

In [2]:
Years = range(2024,2025,1)

predownloaded_directory = "monthly_datasets\\hourly*"
predownloaded_suffix = ".csv" 
predownload_export = "export\\predownload\\"

query_list_file = "New Microsoft Excel Worksheet.xlsx"
query_export = "export\\query\\"

kept_columns = "transition_date, transition_hour, number_of_passage, number_of_passenger, product_kind, transaction_type_desc, town, line_name, station_poi_desc_cd"
spread_columns = "number_of_passage, number_of_passenger"

export_prefix = "hourly_transportation_"
export_suffix = ".parquet"
export_format = "PARQUET"

In [19]:
def clean_dataset(dataset:pd.DataFrame):
    grouping_cols = [col_name.strip(", ") for col_name in kept_columns.split()]
    agging_dict = {col_name:sum for col_name in spread_columns.split(", ")}
    [grouping_cols.remove(col) for col in agging_dict.keys()]

    
        # Sum headcounts, but keep the true location capacity
    df_combined = dataset.groupby(grouping_cols, as_index=False, dropna= False).agg(agging_dict)
    return df_combined

file = pd.read_csv("monthly_datasets\\hourly_transportation_202410.csv")

cleaning_test2 = clean_dataset(file)

3) Gather filtered lists of files

In [ ]:
query_sheet = pd.read_excel(query_list_file)
filtered_query_sheet = query_sheet[query_sheet.Year.isin(Years)]

predownloaded_file_directory_list = [
    f'{predownloaded_directory}{year}*{predownloaded_suffix}' 
    for year in Years
]

4) Query the files and store them in compressed format

In [ ]:
def f(query):
    response = requests.get(query.CSV_URL)

    if response.status_code == 200:
        export_id = f'{query.Year}{query.Month_number:02d}'
        pseudofile = duckdb.read_csv(io.StringIO(response.text))
        duckdb.sql(f"""COPY 
           (SELECT {kept_columns}
 from pseudofile) 
           TO "{query_export}{export_prefix}{export_id}{export_suffix}"
           (FORMAT {export_format})""")
        return True
    else:
        print(f'{query.Month} {query.Year} returned a code of {response.status_code}')
        return False

gean = filtered_query_sheet.tail(3).apply(f, axis=1)

In [ ]:
f"""COPY 
           (SELECT {kept_columns}
 from pseudofile) 
           TO "{query_export}{export_prefix}{22}{export_suffix}" 
           (FORMAT {export_format})"""

duckdb.sql("""
           COPY 
           (SELECT * from cleaning_test2)
           TO "c_test.parquet"
           (FORMAT parquet)
           """)

duckdb.sql(f"""
           COPY 
           (SELECT {kept_columns} from file)
           TO "c_test1.parquet"
           (FORMAT parquet)
           """)

In [ ]:
# duckdb.sql("""
#            SELECT *
#             FROM file
#             WHERE line_name = '34'
#             AND transition_date = '2024-10-04'
#             AND transition_hour = 17
#             AND product_kind = 'UCRETSIZ'
#             AND transaction_type_desc = 'Ucretsiz'
#             AND station_poi_desc_cd = 'ZINCIRLIKUYU 2';
#            """)

grouping_cols = [col_name.strip(", ") for col_name in kept_columns.split()]
agging_dict = {col_name:sum for col_name in spread_columns.split(", ")}
print(list(agging_dict.keys()))
print(grouping_cols)
clean_dataset(file.query(
    "line_name == '34' and "
    "transition_date == '2024-10-04' and "
    "transition_hour == 17 and "
    "product_kind == 'UCRETSIZ' and "
    "transaction_type_desc == 'Ucretsiz' and "
    "station_poi_desc_cd == 'ZINCIRLIKUYU 2'"
)).groupby(grouping_cols, as_index=False, dropna= False).agg(agging_dict)

['number_of_passage', 'number_of_passenger']
['transition_date', 'transition_hour', 'product_kind', 'transaction_type_desc', 'town', 'line_name', 'station_poi_desc_cd']


,transition_date,transition_hour,product_kind,transaction_type_desc,town,line_name,station_poi_desc_cd,number_of_passage,number_of_passenger
0,2024-10-04,17,UCRETSIZ,Ucretsiz,BESIKTAS,34,ZINCIRLIKUYU 2,572,570


In [30]:
s.groupby("A", as_index=False).agg({"B":"sum", "C":"sum"})


,A,B,C
0,0,2,5
1,1,15,18
